<a href="https://colab.research.google.com/github/temesgenaddise/cosc-650-applied-llm-systems/blob/main/Week-6/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week - 6: Advanced Retrieval, Transform and Measure



# Important Libraries

This cell defines the main retrieval settings, including `TOP_K = 5`, `CANDIDATE_K = 10`, and `FINAL_K = 5`, and loads the `all-MiniLM-L6-v2` sentence-transformer model used to create vector embeddings.


In [1]:
%pip install -q sentence-transformers faiss-cpu pandas openai

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 5
CANDIDATE_K = 10
FINAL_K = 5
EVAL_CONFIG = "small (80/20)"

embedder = SentenceTransformer(EMBEDDING_MODEL)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 63.6 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Part 1: Transform and re-rank

Week 5 retriever with at least one query transformation and a re-ranking step over the retrieved candidates.

In [2]:
# Week 5 corpus reused for Week 6.
DOCUMENTS = [
{"doc_id": "D1", "title": "RAG Architecture", "text": """ Retrieval-augmented generation combines a retriever with a language model. At ingestion time, source documents are cleaned, split into chunks, embedded as vectors, and stored in an index. At query time, the same embedding family converts the user question into a vector. The system selects similar chunks and inserts them into a grounded prompt. The generator must answer from that supplied evidence rather than unsupported memory. RAG is useful when knowledge changes frequently or when an answer must be traceable to private sources. Evaluate the retriever and generator separately because fluent generation can hide retrieval mistakes. A production system also records source identifiers, similarity scores, latency, and the exact prompt used for each answer."""},
{"doc_id": "D2", "title": "Chunking Strategies", "text": """ Chunking divides long documents into retrievable units. Fixed-size chunking is simple and predictable, while sentence or section chunking better preserves meaning. Chunk overlap repeats boundary text so a fact near an edge is not separated from its explanation. Very small chunks can improve precision because each result is focused, but they may lose context and increase the number of vectors. Very large chunks preserve context and may improve recall, but they can mix topics and waste the prompt budget. A useful experiment varies chunk size and overlap while keeping the embedding model, query set, and top-k constant. Metadata such as document ID, heading, and character offsets should travel with every chunk."""},
{"doc_id": "D3", "title": "Embeddings and Similarity", "text": """ An embedding model maps text to a dense numerical vector. Semantically related passages should be close even when they use different vocabulary. Sentence-transformer models can run locally and are commonly used for RAG prototypes. The same model and normalization procedure must encode both corpus chunks and queries. With normalized vectors, the inner product equals cosine similarity. Domain-specific language can challenge a general-purpose embedding model, so compare models using a labeled query set. Embedding quality, vector dimension, speed, memory use, and licensing all matter when selecting a model."""},
{"doc_id": "D4", "title": "Vector Indexes and FAISS", "text": """ A vector index stores embeddings and searches for nearest neighbors. FAISS provides exact and approximate index types. IndexFlatIP performs exact inner-product search and suits a small evaluation corpus. Approximate indexes such as HNSW or IVF trade a small amount of recall for lower latency at large scale. Vectors must use float32 for common FAISS indexes. Returned row positions should be mapped back to chunk text and metadata. Similarity scores rank candidates but are not calibrated probabilities. Filters can restrict retrieval by tenant, date, language, or document type before or after vector search."""},
{"doc_id": "D5", "title": "Retrieval Evaluation", "text": """ Retrieval evaluation starts with a test set created independently of system output. For each query, evaluators decide which chunks satisfy the information need. Precision at k is the fraction of the k retrieved chunks that are relevant. Recall at k is the fraction of all relevant chunks that appear in the top k. Precision rewards focused results, while recall rewards coverage. Mean precision and mean recall summarize performance across queries, but per-query results reveal failures. Chunk-level labels must be regenerated consistently when chunk boundaries change. Other useful measures include hit rate, mean reciprocal rank, and normalized discounted cumulative gain. Evaluation should report top-k and the labeling rule."""},
{"doc_id": "D6", "title": "Grounded Generation", "text": """The generation stage assembles retrieved passages into a prompt. A grounded prompt tells the model to use only the provided context, cite source labels, and admit when evidence is insufficient. The context should include chunk identifiers so claims can be traced. Prompt length is limited, so irrelevant large chunks can crowd out useful evidence. Low temperature improves repeatability but does not guarantee factuality. The generator must not treat a retrieval score as proof that a passage is correct. Answer quality can be evaluated for correctness, faithfulness to context, completeness, and citation accuracy."""},
{"doc_id": "D7", "title": "Failure Modes and Mitigations", "text": """ RAG failures can occur during ingestion, retrieval, or generation. A boundary failure happens when chunking separates a key term from the sentence that explains it. Vocabulary mismatch occurs when the query and relevant passage use different words. Similar but wrong passages can rank highly when they share many terms with the query. Mitigations include overlap, semantic chunking, query rewriting, hybrid lexical and vector search, metadata filtering, reranking, and a larger candidate pool. If evidence is weak, the system should abstain rather than guess. Failure analysis should show the query, retrieved chunks, expected evidence, cause, and a targeted mitigation."""},
{"doc_id": "D8", "title": "Connecting Retrieval to Answer Quality", "text": """ Answer quality depends on both evidence availability and generator behavior. If no relevant chunk reaches the prompt, a grounded generator cannot produce a complete supported answer. High retrieval recall creates an upper bound on answer coverage, while high precision reduces distracting context. To connect the stages, run generation with results from each retrieval configuration and score whether required facts appear and whether citations point to relevant chunks. A controlled comparison keeps the question and generation settings fixed. Retrieval metrics and answer metrics can disagree: a single excellent chunk may support a correct answer even when precision at k is low, while several partially relevant chunks may still omit the decisive fact."""}
]
docs_df = pd.DataFrame(DOCUMENTS)
#display(docs_df[["doc_id", "title"]])
print(f"Corpus: {len(DOCUMENTS)} documents")

Corpus: 8 documents


This cell builds the Week 5 baseline retrieval system. It chunks each document using an 80-word chunk size with 20-word overlap, embeds the chunks with the sentence-transformer model, stores the normalized vectors in a FAISS inner-product index, and defines the `retrieve()` function used for similarity search.


In [3]:
CHUNK_SIZE = 80
CHUNK_OVERLAP = 20

def chunk_document(doc, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = doc["text"].split()
    chunks = []
    step = size - overlap

    for start in range(0, len(words), step):
        piece = words[start:start + size]
        if not piece:
            break

        chunks.append({
            "chunk_id": f'{doc["doc_id"]}_w{start:04d}',
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "start_word": start,
            "end_word": start + len(piece),
            "text": " ".join(piece)
        })

        if start + size >= len(words):
            break

    return chunks

def build_chunks():
    return [
        chunk
        for doc in DOCUMENTS
        for chunk in chunk_document(doc)
    ]

chunk_sets = {EVAL_CONFIG: build_chunks()}

def build_index(chunks):
    vectors = embedder.encode(
        [c["text"] for c in chunks],
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index

indexes = {EVAL_CONFIG: build_index(chunk_sets[EVAL_CONFIG])}

def retrieve(query, config_name=EVAL_CONFIG, k=TOP_K):
    q = embedder.encode(
        [query],
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    chunks = chunk_sets[config_name]
    scores, positions = indexes[config_name].search(
        q, min(k, len(chunks))
    )

    return [
        {
            **chunks[i],
            "score": float(score),
            "rank": rank + 1
        }
        for rank, (i, score) in enumerate(zip(positions[0], scores[0]))
        if i >= 0
    ]

print(f"Week-5 configuration retained for Week 6: {EVAL_CONFIG}")
print(f"Number of chunks: {len(chunk_sets[EVAL_CONFIG])}")

Week-5 configuration retained for Week 6: small (80/20)
Number of chunks: 16


     The output confirms that the retained configuration is small (80/20)
     and that the eight documents produced 16 chunks. These 16 chunks are
     the searchable units stored in the FAISS index.


This cell defines the 10 evaluation questions and the expected relevant document for each question. It also specifies required anchor terms and defines `is_relevant()`, which marks a retrieved chunk as relevant only when it comes from the expected document and contains all required anchor terms.


In [4]:
TEST_QUERIES = [
 {"qid": "Q1", "query": "What are the five main stages of a RAG pipeline?", "doc_id": "D1", "anchors":["ingestion", "chunks", "embedded", "index"]},
 {"qid": "Q2", "query": "Why is overlap used when splitting documents?", "doc_id":"D2", "anchors":["overlap", "boundary"]},
 {"qid": "Q3", "query": "When is inner product equivalent to cosine similarity?", "doc_id": "D3", "anchors":["normalized", "inner product", "cosine"]},
 {"qid": "Q4", "query": "Which FAISS index performs exact inner-product search?", "doc_id":"D4", "anchors":["indexflatip", "exact"]},
 {"qid": "Q5", "query": "How are precision at k and recall at k defined?", "doc_id": "D5", "anchors":["precision at k", "recall at k"]},
 {"qid": "Q6", "query": "What instructions should a grounded generation prompt include?", "doc_id": "D6", "anchors":["only", "context", "cite", "insufficient"]},
 {"qid": "Q7", "query": "What is a chunk-boundary failure and how can it be mitigated?", "doc_id":"D7", "anchors":["boundary", "overlap"]},
 {"qid": "Q8", "query": "How does retrieval recall affect possible answer coverage?", "doc_id":"D8", "anchors":["recall", "upper bound", "coverage"]},
 {"qid": "Q9", "query": "Why should similarity scores not be interpreted as probabilities?", "doc_id":"D4", "anchors":["scores", "probabilities"]},
 {"qid": "Q10", "query": "Why can retrieval metrics and answer metrics disagree?", "doc_id":"D8", "anchors":["metrics", "disagree", "single excellent chunk"]},
]

def is_relevant(chunk, case):
    text = chunk["text"].lower()
    return chunk["doc_id"] == case["doc_id"] and all(a.lower() in text for a in case["anchors"])

# Part 2: Measure the effect


This cell implements selective query transformation. Only Q7 and Q9 are expanded with additional retrieval terms intended to reduce vocabulary mismatch or strengthen the retrieval signal. The remaining queries are left unchanged so the effect of selective transformation can be compared with the baseline.


In [5]:
# Selective query transformation: only Q7 and Q9 are expanded.
# Only these two Week-5 queries are transformed.
SELECTED_TRANSFORMATIONS = {
    "Q7": "boundary failure chunk overlap semantic chunking mitigation",
    "Q9": "similarity scores ranking calibration probabilities"
}

TRANSFORMED_QIDS = set(SELECTED_TRANSFORMATIONS)


def transform_query(case):
    """Transform only selected queries; return all others unchanged."""
    expansion = SELECTED_TRANSFORMATIONS.get(case["qid"])
    if expansion is None:
        return case["query"]
    return f'{case["query"]} {expansion}'.strip()


transformation_preview = pd.DataFrame([
    {
        "qid": case["qid"],
        "transformed": case["qid"] in TRANSFORMED_QIDS,
        "original_query": case["query"],
        "retrieval_query": transform_query(case)
    }
    for case in TEST_QUERIES
])

#display(transformation_preview)
print(f"Queries transformed: {sorted(TRANSFORMED_QIDS)}")
print(f"Number transformed: {len(TRANSFORMED_QIDS)} of {len(TEST_QUERIES)}")

Queries transformed: ['Q7', 'Q9']
Number transformed: 2 of 10


     The output confirms that only Q7 and Q9 were transformed— 2 of the 10 queries.
     The displayed table shows the original query beside the retrieval query, making the exact expansion terms visible.


This cell loads a cross-encoder reranking model and defines the two-stage retrieval function. FAISS first retrieves a larger candidate set, and then the cross-encoder scores each candidate against the original user query. The candidates are reordered by the cross-encoder score, and the top five are returned.


In [6]:
# Re-rank a larger candidate pool for every query.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def retrieve_and_rerank(
    case,
    config_name=EVAL_CONFIG,
    candidate_k=CANDIDATE_K,
    final_k=FINAL_K
):
    # Stage 1: transform only Q7/Q9; all other queries remain unchanged.
    retrieval_query = transform_query(case)
    was_transformed = case["qid"] in TRANSFORMED_QIDS

    # Retrieve a larger candidate pool from the Week-5 FAISS retriever.
    candidates = retrieve(
        retrieval_query,
        config_name,
        k=candidate_k
    )

    # Stage 2: re-rank every candidate pool using the ORIGINAL query.
    pairs = [
        [case["query"], candidate["text"]]
        for candidate in candidates
    ]
    rerank_scores = reranker.predict(pairs)

    reranked = []
    for candidate, rerank_score in zip(candidates, rerank_scores):
        item = candidate.copy()
        item["retrieval_query"] = retrieval_query
        item["query_transformed"] = was_transformed
        item["first_stage_rank"] = candidate["rank"]
        item["rerank_score"] = float(rerank_score)
        reranked.append(item)

    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)

    for rank, item in enumerate(reranked, start=1):
        item["rerank_rank"] = rank
        item["rank"] = rank

    return reranked[:final_k]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

This cell inspects Q7 in detail. It runs the transformed Q7 query through first-stage FAISS retrieval and cross-encoder reranking, then displays the candidate chunks, their original retrieval ranks, reranked positions, scores, and relevance labels.


In [7]:
# Inspect one transformed query so both stages are visible.
example_case = next(case for case in TEST_QUERIES if case["qid"] == "Q7")
example_hits = retrieve_and_rerank(example_case)

display(pd.DataFrame([
    {
        "qid": example_case["qid"],
        "query_transformed": h["query_transformed"],
        "chunk_id": h["chunk_id"],
        "doc_id": h["doc_id"],
        "first_stage_rank": h["first_stage_rank"],
        "rerank_rank": h["rerank_rank"],
        "rerank_score": h["rerank_score"],
        "relevant": is_relevant(h, example_case)
    }
    for h in example_hits
]))

,qid,query_transformed,chunk_id,doc_id,first_stage_rank,rerank_rank,rerank_score,relevant
0,Q7,True,D7_w0000,D7,1,1,3.996496,True
1,Q7,True,D5_w0060,D5,5,2,-1.059086,False
2,Q7,True,D2_w0000,D2,2,3,-2.159733,False
3,Q7,True,D7_w0060,D7,4,4,-3.933040,False
4,Q7,True,D2_w0060,D2,3,5,-8.545666,False



     For Q7, `D7_w0000` remains at rank 1 after reranking and is marked
     relevant. Some other chunks may come from the expected document but still
     be marked irrelevant because the notebook requires all specified anchor
     terms to occur in the chunk. Negative cross-encoder scores are not
     errors; the scores are used comparatively for ranking.


This cell evaluates all 10 queries under two conditions: the original Week 5 retrieval method and the Week 6 transformed/reranked method. For each query, it calculates Precision@5, Recall@5, and Mean Reciprocal Rank (MRR), then computes the change in each metric.


In [8]:
# Measure baseline -> transformed/reranked retrieval.
def reciprocal_rank(case, hits):
    for rank, hit in enumerate(hits, start=1):
        if is_relevant(hit, case):
            return 1.0 / rank
    return 0.0


comparison_rows = []

for case in TEST_QUERIES:
    # BEFORE: original Week-5 retriever.
    baseline_hits = retrieve(case["query"], EVAL_CONFIG, k=FINAL_K)
    baseline_relevant = sum(is_relevant(hit, case) for hit in baseline_hits)

    total_relevant = sum(
        is_relevant(chunk, case)
        for chunk in chunk_sets[EVAL_CONFIG]
    )

    baseline_precision = baseline_relevant / len(baseline_hits) if baseline_hits else 0.0
    baseline_recall = baseline_relevant / total_relevant if total_relevant else 0.0
    baseline_mrr = reciprocal_rank(case, baseline_hits)

     # AFTER: selective query transformation + candidate re-ranking.
    improved_hits = retrieve_and_rerank(case, EVAL_CONFIG)
    improved_relevant = sum(is_relevant(hit, case) for hit in improved_hits)
    improved_precision = improved_relevant / len(improved_hits) if improved_hits else 0.0
    improved_recall = improved_relevant / total_relevant if total_relevant else 0.0
    improved_mrr = reciprocal_rank(case, improved_hits)

    comparison_rows.append({
        "qid": case["qid"],
        "query": case["query"],
        "transformed": case["qid"] in TRANSFORMED_QIDS,
        "retrieval_query_after": transform_query(case),
        "before_precision@5": baseline_precision,
        "after_precision@5": improved_precision,
        "delta_precision@5": improved_precision - baseline_precision,
        "before_recall@5": baseline_recall,
        "after_recall@5": improved_recall,
        "delta_recall@5": improved_recall - baseline_recall,
        "before_mrr": baseline_mrr,
        "after_mrr": improved_mrr,
        "delta_mrr": improved_mrr - baseline_mrr
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(3))

,qid,query,transformed,retrieval_query_after,before_precision@5,after_precision@5,delta_precision@5,before_recall@5,after_recall@5,delta_recall@5,before_mrr,after_mrr,delta_mrr
0,Q1,What are the five main stages of a RAG pipeline?,False,What are the five main stages of a RAG pipeline?,0.2,0.2,0.0,1.0,1.0,0.0,0.25,0.5,0.25
1,Q2,Why is overlap used when splitting documents?,False,Why is overlap used when splitting documents?,0.2,0.2,0.0,1.0,1.0,0.0,1.00,1.0,0.00
2,Q3,When is inner product equivalent to cosine sim...,False,When is inner product equivalent to cosine sim...,0.2,0.2,0.0,1.0,1.0,0.0,1.00,1.0,0.00
3,Q4,Which FAISS index performs exact inner-product...,False,Which FAISS index performs exact inner-product...,0.2,0.2,0.0,1.0,1.0,0.0,1.00,1.0,0.00
4,Q5,How are precision at k and recall at k defined?,False,How are precision at k and recall at k defined?,0.2,0.2,0.0,1.0,1.0,0.0,1.00,1.0,0.00
5,Q6,What instructions should a grounded generation...,False,What instructions should a grounded generation...,0.2,0.2,0.0,1.0,1.0,0.0,1.00,1.0,0.00
6,Q7,What is a chunk-boundary failure and how can i...,True,What is a chunk-boundary failure and how can i...,0.2,0.2,0.0,1.0,1.0,0.0,1.00,1.0,0.00
7,Q8,How does retrieval recall affect possible answ...,False,How does retrieval recall affect possible answ...,0.2,0.2,0.0,1.0,1.0,0.0,1.00,1.0,0.00
8,Q9,Why should similarity scores not be interprete...,True,Why should similarity scores not be interprete...,0.4,0.4,0.0,1.0,1.0,0.0,1.00,1.0,0.00
9,Q10,Why can retrieval metrics and answer metrics d...,False,Why can retrieval metrics and answer metrics d...,0.2,0.2,0.0,1.0,1.0,0.0,1.00,1.0,0.00


### Result Explanation
     The main visible improvement occurs for Q1, whose MRR increases from
     0.25 to 0.50. This means the first relevant result moved from rank 4 to
     rank 2. Precision@5 and Recall@5 remain unchanged, so the intervention
     improved ranking position rather than the number of relevant chunks retrieved.



This cell summarizes the experiment across all queries. It calculates average Precision@5, Recall@5, and MRR before and after the intervention and classifies each query as benefited, unchanged, or regressed based on the metric changes.


In [9]:
print("Summary comparison")
print(" ")
summary_comparison = pd.DataFrame({
    "Before": [
        comparison_df["before_precision@5"].mean(),
        comparison_df["before_recall@5"].mean(),
        comparison_df["before_mrr"].mean()
    ],
    "After": [
        comparison_df["after_precision@5"].mean(),
        comparison_df["after_recall@5"].mean(),
        comparison_df["after_mrr"].mean()
    ]
}, index=[
    "Mean Precision@5",
    "Mean Recall@5",
    "Mean MRR"
])

summary_comparison["Delta"] = (
    summary_comparison["After"]
    - summary_comparison["Before"]
)

display(summary_comparison.round(3))
print("\n")

print("Comparsion by query")
print(" ")

def classify_effect(delta, tolerance=1e-12):
    if delta > tolerance:
        return "Benefited"
    elif delta < -tolerance:
        return "Regressed"
    return "No change"


comparison_df["effect"] = comparison_df["delta_mrr"].apply(classify_effect)

effect_table = comparison_df[
    [
        "qid", "query", "transformed",
        "before_mrr", "after_mrr", "delta_mrr",
        "before_recall@5", "after_recall@5", "effect"
    ]
].copy()

display(effect_table.round(3))
print("\n")

print("Query counts by effect:")
print(" ")
display(
    comparison_df["effect"]
    .value_counts()
    .rename_axis("effect")
    .reset_index(name="number_of_queries")
)
print("\n")

print("Effect by transformation status:")
print(" ")
display(
    comparison_df.groupby("transformed")
    .agg(
        queries=("qid", "count"),
        mean_delta_precision_at_5=("delta_precision@5", "mean"),
        mean_delta_recall_at_5=("delta_recall@5", "mean"),
        mean_delta_mrr=("delta_mrr", "mean")
    )
    .round(3)
)

Summary comparison
 


,Before,After,Delta
Mean Precision@5,0.220,0.22,0.000
Mean Recall@5,1.000,1.00,0.000
Mean MRR,0.925,0.95,0.025




Comparsion by query
 


,qid,query,transformed,before_mrr,after_mrr,delta_mrr,before_recall@5,after_recall@5,effect
0,Q1,What are the five main stages of a RAG pipeline?,False,0.25,0.5,0.25,1.0,1.0,Benefited
1,Q2,Why is overlap used when splitting documents?,False,1.00,1.0,0.00,1.0,1.0,No change
2,Q3,When is inner product equivalent to cosine sim...,False,1.00,1.0,0.00,1.0,1.0,No change
3,Q4,Which FAISS index performs exact inner-product...,False,1.00,1.0,0.00,1.0,1.0,No change
4,Q5,How are precision at k and recall at k defined?,False,1.00,1.0,0.00,1.0,1.0,No change
5,Q6,What instructions should a grounded generation...,False,1.00,1.0,0.00,1.0,1.0,No change
6,Q7,What is a chunk-boundary failure and how can i...,True,1.00,1.0,0.00,1.0,1.0,No change
7,Q8,How does retrieval recall affect possible answ...,False,1.00,1.0,0.00,1.0,1.0,No change
8,Q9,Why should similarity scores not be interprete...,True,1.00,1.0,0.00,1.0,1.0,No change
9,Q10,Why can retrieval metrics and answer metrics d...,False,1.00,1.0,0.00,1.0,1.0,No change




Query counts by effect:
 


,effect,number_of_queries
0,No change,9
1,Benefited,1




Effect by transformation status:
 


,queries,mean_delta_precision_at_5,mean_delta_recall_at_5,mean_delta_mrr
transformed,,,,
False,8,0.0,0.0,0.031
True,2,0.0,0.0,0.000


### Result Explanation
     Mean Precision@5 remains 0.220, mean Recall@5 remains 1.000, and
     mean MRR improves from 0.925 to 0.950, a gain of 0.025. The
     summary shows 1 benefited query, 9 unchanged queries, and 0
     regressions. Because Q7 and Q9 themselves show no MRR improvement, the
     measured gain comes from reranking rather than from those explicit query
      transformations.


This cell creates a concise per-query outcome summary. It identifies which queries benefited, which showed no measurable change, and whether any query regressed after transformation and reranking.


In [10]:
print("\nOverall before -> after:")
display(summary_comparison.round(3))
print("\n")

print("\nPer-query effect:")
display(effect_table.round(3))
print("\n")

benefited = comparison_df.loc[comparison_df["effect"] == "Benefited", "qid"].tolist()
unchanged = comparison_df.loc[comparison_df["effect"] == "No change", "qid"].tolist()
regressed = comparison_df.loc[comparison_df["effect"] == "Regressed", "qid"].tolist()
print("\n")
print("Benefited:", benefited if benefited else "None")
print("No change:", unchanged if unchanged else "None")
print("Regressed:", regressed if regressed else "None")


Overall before -> after:


,Before,After,Delta
Mean Precision@5,0.220,0.22,0.000
Mean Recall@5,1.000,1.00,0.000
Mean MRR,0.925,0.95,0.025





Per-query effect:


,qid,query,transformed,before_mrr,after_mrr,delta_mrr,before_recall@5,after_recall@5,effect
0,Q1,What are the five main stages of a RAG pipeline?,False,0.25,0.5,0.25,1.0,1.0,Benefited
1,Q2,Why is overlap used when splitting documents?,False,1.00,1.0,0.00,1.0,1.0,No change
2,Q3,When is inner product equivalent to cosine sim...,False,1.00,1.0,0.00,1.0,1.0,No change
3,Q4,Which FAISS index performs exact inner-product...,False,1.00,1.0,0.00,1.0,1.0,No change
4,Q5,How are precision at k and recall at k defined?,False,1.00,1.0,0.00,1.0,1.0,No change
5,Q6,What instructions should a grounded generation...,False,1.00,1.0,0.00,1.0,1.0,No change
6,Q7,What is a chunk-boundary failure and how can i...,True,1.00,1.0,0.00,1.0,1.0,No change
7,Q8,How does retrieval recall affect possible answ...,False,1.00,1.0,0.00,1.0,1.0,No change
8,Q9,Why should similarity scores not be interprete...,True,1.00,1.0,0.00,1.0,1.0,No change
9,Q10,Why can retrieval metrics and answer metrics d...,False,1.00,1.0,0.00,1.0,1.0,No change






Benefited: ['Q1']
No change: ['Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10']
Regressed: None


### Result Explanation
     The output reports Q1 as benefited, Q2–Q10 as unchanged, and no
     regressed queries. This provides a simple final view of the measured
     effect of the Week 6 retrieval changes.


## HyDE

This HyDE cell connects to Gemini through an OpenAI-compatible client. The `hyde()` function asks the model to generate a one-sentence hypothetical answer to a retrieval question. The generated text demonstrates the document-generation step of Hypothetical Document Embeddings (HyDE).

In [11]:
from openai import OpenAI
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError("Add GEMINI_API_KEY to Colab Secrets before running the notebook.")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
MODEL = "gemini-3.8-flash"
print("Gemini client ready. Model:", MODEL)

def hyde(q, model=MODEL):
    return client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': 'Write a one-sentence answer to: ' + q}]
    ).choices[0].message.content

print(hyde('What is a chunk-boundary failure and how can it be mitigated?'))

Gemini client ready. Model: gemini-3.8-flash
A chunk-boundary failure occurs when critical context is fragmented across adjacent text segments during data splitting, which can be mitigated by implementing overlapping chunk windows or semantic-aware chunking strategies.


### Result Explanation
     The generated hypothetical answer states that chunk-boundary problems
     occur when relevant context is divided across adjacent chunks and suggests
     overlap or semantic/sentence-aware chunking as mitigation. This cell
     demonstrates the generation step of HyDE; it does not yet embed the
     generated answer and evaluate a full HyDE retrieval pipeline.

# Part 3: Find one failure and explain it


     No query showed worse retrieval performance after the transformation and
     reranking process in this experiment. In particular, the two transformed
     queries, Q7 and Q9, maintained the same Precision@5, Recall@5, and MRR as
     their baseline results.This suggests that the query transformations held
     across the selected test cases. The added terms were closely aligned
     with the original information needs. For example, Q7 was expanded with
     terms related to boundary  failure, chunk overlap, semantic chunking,
     and mitigation, while Q9 added terms related to similarity scores,
     ranking, calibration, and probabilities. These terms reinforced
     concepts already present in the relevant documents rather than changing
     the meaning of the original queries.



## Part 4 : (optional enrichment): schema-constrained extraction






This cell defines a small typed entity schema for structured extraction. The schema contains retrieval failure types, mitigation strategies, and retrieval concepts. It then scans each chunk for these predefined phrases so the notebook can answer structured questions such as counts and relationships.


In [ ]:
ENTITY_SCHEMA = {
    "failure_type": [
        "boundary failure",
        "vocabulary mismatch"
    ],
    "mitigation": [
        "overlap",
        "semantic chunking",
        "query rewriting",
        "hybrid lexical and vector search",
        "metadata filtering",
        "reranking",
        "larger candidate pool"
    ],
    "retrieval_concept": [
        "precision",
        "recall",
        "similarity scores",
        "cosine similarity",
        "inner product"
    ]
}

def extract_typed_entities(text, schema=ENTITY_SCHEMA):
    text_lower = text.lower()
    return {
        entity_type: [value for value in values if value.lower() in text_lower]
        for entity_type, values in schema.items()
    }

typed_records = []
for chunk in chunk_sets[EVAL_CONFIG]:
    typed_records.append({
        "chunk_id": chunk["chunk_id"],
        "doc_id": chunk["doc_id"],
        "title": chunk["title"],
        "entities": extract_typed_entities(chunk["text"])
    })

typed_df = pd.DataFrame([
    {
        "chunk_id": r["chunk_id"],
        "doc_id": r["doc_id"],
        "title": r["title"],
        "failure_type": ", ".join(r["entities"]["failure_type"]),
        "mitigation": ", ".join(r["entities"]["mitigation"]),
        "retrieval_concept": ", ".join(r["entities"]["retrieval_concept"])
    }
    for r in typed_records
])

print("Typed entities extracted from corpus chunks:")
display(typed_df[
    (typed_df["failure_type"] != "") |
    (typed_df["mitigation"] != "") |
    (typed_df["retrieval_concept"] != "")
])


boundary_chunks = [
    r for r in typed_records
    if "boundary failure" in r["entities"]["failure_type"]
]

print("\nStructured query 1: How many chunks explicitly mention 'boundary failure'?")
print("Count:", len(boundary_chunks))
print("Matching chunk IDs:", [r["chunk_id"] for r in boundary_chunks])


mitigations_for_boundary_failure = sorted({
    mitigation
    for r in boundary_chunks
    for mitigation in r["entities"]["mitigation"]
})

print("\nStructured query 2: Which typed mitigations are associated with boundary-failure chunks?")
print("Mitigations:", mitigations_for_boundary_failure)

# Make sure it is ready for Review......//////

Typed entities extracted from corpus chunks:


,chunk_id,doc_id,title,failure_type,mitigation,retrieval_concept
1,D1_w0060,D1,RAG Architecture,,,similarity scores
2,D2_w0000,D2,Chunking Strategies,,overlap,"precision, recall"
3,D2_w0060,D2,Chunking Strategies,,overlap,recall
4,D3_w0000,D3,Embeddings and Similarity,,,"cosine similarity, inner product"
6,D4_w0000,D4,Vector Indexes and FAISS,,,"recall, similarity scores"
7,D4_w0060,D4,Vector Indexes and FAISS,,,similarity scores
8,D5_w0000,D5,Retrieval Evaluation,,,"precision, recall"
9,D5_w0060,D5,Retrieval Evaluation,,,"precision, recall"
12,D7_w0000,D7,Failure Modes and Mitigations,"boundary failure, vocabulary mismatch","overlap, semantic chunking, query rewriting, h...",
13,D7_w0060,D7,Failure Modes and Mitigations,,"metadata filtering, reranking, larger candidat...",



Structured query 1: How many chunks explicitly mention 'boundary failure'?
Count: 1
Matching chunk IDs: ['D7_w0000']

Structured query 2: Which typed mitigations are associated with boundary-failure chunks?
Mitigations: ['hybrid lexical and vector search', 'larger candidate pool', 'metadata filtering', 'overlap', 'query rewriting', 'reranking', 'semantic chunking']


### Interpretation

     The typed-extraction layer complements vector retrieval rather than
     replacing it. Vector search ranks chunks by semantic similarity, but a
     similarity score does not directly provide an exact count or an explicit
     entity relationship. Here, the schema converts selected corpus concepts
     into typed fields. The first structured query can therefore count chunks
     that explicitly contain the **boundary failure** entity. The second
     query performs a small multi-hop lookup by first finding those
     failure-related chunks and then collecting the **mitigation** entities
     extracted from the same evidence.



